In [11]:
import pandas as pd
import re
from sqlalchemy import create_engine
import urllib

In [12]:
# Configurar cadena de conexión
params = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=181.57.189.150,1433;"
    "DATABASE=Ventas_Comerssia;"
    "UID=sa;"
    "PWD=P@ssw0rd*;"
)

# Crear el motor
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

In [13]:
query = """
SELECT DISTINCT c.Cliente,
	c.Documento,
    c.Nombres,
    c.Apellidos,
    c.Email,
    c.Celular,
    c.TipoIdentificacion
FROM Ventas_Comerssia.DBO.View_Clientes c
INNER JOIN Ventas_Comerssia.dbo.Ventas_Unificadas v ON c.Cliente = v.Cliente
WHERE v.Fecha >= '2025-08-24'
"""

# Ejecutar y cargar en DataFrame
df_clientes = pd.read_sql(query, engine)

df_clientes = df_clientes.drop_duplicates(subset='Cliente', keep='first')

# Obtener los CLICodigos ya existentes en la tabla SQL
clientes_existentes = pd.read_sql("SELECT Cliente FROM View_Contacto_Clientes", con=engine)

In [14]:
#Limpiar Celular
def limpiar_celular(cel):
    if pd.isna(cel):
        return ""
    cel = re.sub(r'\D', '', str(cel))  # Elimina todo lo que no sea dígito
    if cel.startswith("57") and len(cel) > 10:
        cel = cel[2:]
    return cel

df_clientes['Celular'] = df_clientes['Celular'].apply(limpiar_celular)

#Validar celular
def es_celular_valido(cel):
    return bool(re.fullmatch(r"3\d{9}", str(cel)))

df_clientes['CelularValido'] = df_clientes['Celular'].apply(es_celular_valido)

#Limpiar y validar email
df_clientes['Email'] = df_clientes['Email'].apply(lambda x: str(x).strip())

def es_email_valido(email):
    email = str(email).strip()

    if pd.isna(email):
        return False

    email = str(email).strip()

    # Si está vacío
    if not email:
        return False
    
    # Palabras prohibidas
    palabras_no_permitidas = ["NEGAD", "PENDI", "negad","pendi"]
    if any(palabra in email for palabra in palabras_no_permitidas):
        return False
    
    # Validación de formato
    return bool(re.fullmatch(r'^[\w\.-]+@[\w\.-]+\.\w+$', email))

df_clientes['EmailValido'] = df_clientes['Email'].apply(es_email_valido)

In [15]:
# Detectar duplicados
cel_duplicados = df_clientes['Celular'].duplicated(keep=False)
email_duplicados = df_clientes['Email'].duplicated(keep=False)

# Marcar celulares duplicados como no válidos
df_clientes.loc[cel_duplicados, 'CelularValido'] = False

# Marcar correos duplicados como no válidos
df_clientes.loc[email_duplicados, 'EmailValido'] = False

In [16]:
#Agregar columna contacto
def determinar_contacto(row):
    if row['CelularValido'] and row['EmailValido']:
        return "Cel + Email"
    elif row['CelularValido']:
        return "Cel"
    elif row['EmailValido']:
        return "Email"
    else:
        return "Falso"

df_clientes['Contacto'] = df_clientes.apply(determinar_contacto, axis=1)

In [17]:
# # Filtrar para dejar solo nuevos clientes
df_nuevos = df_clientes[~df_clientes["Cliente"].isin(clientes_existentes["Cliente"])]

In [18]:
# Insertar solo los nuevos en SQL
if not df_nuevos.empty:
    df_nuevos.to_sql("View_Contacto_Clientes", con=engine, if_exists="append", index=False)
    print(f"{len(df_nuevos)} nuevos clientes insertados.")
else:
    print("No hay nuevos clientes por insertar.")

63 nuevos clientes insertados.


In [19]:
# Exportar todos los registros con validaciones incluidas
# df_clientes.to_excel("clientes_con_validacion.xlsx", index=False)

In [20]:
# df_clientes.to_sql("View_Contacto_Clientes", engine, if_exists="replace", index=False)